# Test evaluation

## 1. Final scores on the test set
All choices were made on validation. I evaluate every model once on the test set (January 2024 to August 2026) and change nothing afterwards.
The Elo logistic and boosting models are refitted on train + validation with the settings chosen on validation. Poisson and Dixon-Coles are refitted every month on all earlier matches, as before.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src.database import ROOT, query
from src.evaluation import OUTCOMES, compare, paired_difference
from src.models.boosting import BoostingModel
from src.models.dixon_coles import DixonColesModel
from src.models.elo_logistic import EloLogistic
from src.models.poisson import PoissonModel, rolling_predict
from src.splits import load_matches

expected_goals = pd.read_csv(ROOT / "data" / "processed" / "poisson_expected_goals.csv")
matches = load_matches().merge(expected_goals, on="match_id")
fit_set = matches[matches["split"] != "test"]
test = matches[matches["split"] == "test"]
history = query("""
    SELECT date, home_team, away_team, home_score, away_score, neutral
    FROM clean_results
    ORDER BY date, match_id
""")
len(fit_set), len(test)

(21377, 2583)

In [2]:
frequencies = fit_set.groupby("neutral")["result"].value_counts(normalize=True).unstack()[OUTCOMES]
stacking_features = ["elo_diff", "home_venue", "poisson_home_goals", "poisson_away_goals"]

test_probs = {
    "uniform": np.full((len(test), 3), 1 / 3),
    "frequency_by_venue": frequencies.reindex(test["neutral"]).to_numpy(),
    "elo_logistic": EloLogistic().fit(fit_set).predict_proba(test),
    "boosting": BoostingModel(max_iter=50, max_depth=4).fit(fit_set).predict_proba(test),
    "poisson": rolling_predict(lambda: PoissonModel(), history, test),
    "dixon_coles": rolling_predict(lambda: DixonColesModel(), history, test),
    "stacking": BoostingModel(max_iter=50, max_depth=4, features=stacking_features).fit(fit_set).predict_proba(test),
}
compare(test_probs, test["result"])

,log_loss,brier,rps
uniform,1.0986,0.6667,0.2372
frequency_by_venue,1.0538,0.6352,0.2256
elo_logistic,0.8649,0.5098,0.1658
boosting,0.8652,0.5091,0.1657
poisson,0.8448,0.4963,0.1602
dixon_coles,0.8432,0.4957,0.1601
stacking,0.8478,0.4983,0.1610


In [3]:
pairs = [("poisson", "elo_logistic"), ("stacking", "poisson"), ("boosting", "elo_logistic")]
pd.DataFrame(
    [{"comparison": f"{a} - {b}", **paired_difference(test_probs[a], test_probs[b], test["result"])} for a, b in pairs]
).round(4)

,comparison,mean_diff,std_error
0,poisson - elo_logistic,-0.0202,0.0048
1,stacking - poisson,0.0030,0.0027
2,boosting - elo_logistic,0.0003,0.0027


The test confirms the validation ranking. Poisson beats the Elo baseline by 0.0202 in log loss (4.2 standard errors), a clear gap that was only borderline on validation. Boosting alone adds nothing to the Elo baseline. Stacking and Dixon-Coles are within noise of Poisson, so the simplest model stays the right choice.
All scores are better than on validation, for every model including the baselines: 2024-2026 matches were slightly easier to predict. The ranking of models is what matters.